Cell M3-1(markdown)|notebook 定位

### 03 | 假設檢定(M3)

**目的**:把 M2 切片得到的「看起來有差」升級為「統計上站得住的有差」,並且區分「統計上有差」與「實務上值得行動」。M2 的切片是描述性的:它呈現各客群的流失率,並由分析者判斷形狀與樣本是否足以支持假設;但描述性數字無法回答「這個差距是真的存在,還是這一萬名客戶剛好抽到這樣」。本模組以假設檢定回答此問題,並以效果量回答「差距有多大」,讓後續 M4(模型特徵)、M5(名單條件與策略)、M8(對決策者的報告)能建立在推論性的證據上,而非建立在描述性的觀察上。

**支持的決策**:問題定義書 §2 的決策 1(挽留資源應優先投放在哪些客群)與決策 2(高風險客戶的可辨識訊號是什麼)。每一個檢定在本 notebook 第 1 節都寫明它支持哪一個後續決策,以及若檢定結果不顯著,該決策會受到什麼影響。

**分析順序的定位**:本專案的分析流程是「先確定要支持的決策(問題定義書)→ 依業務知識事前指定假設(M2 假設樹)→ 以切片探索資料(M2)→ 以檢定確認證據站得住並量化差距(本模組)→ 以確認過的證據決定策略內容(M5)→ 以實驗驗證策略有效(M6)」。本模組確認的是「證據」,不是「策略有效」;策略是否有效(例如深化往來是否真能降低流失),只有 M6 的實驗設計能回答,因為觀察性資料只能建立關聯、無法確認因果。

**資料來源**:`clean.customers`(M1 產出,10,127 列,基準流失率 16.07%)。檢定所需的欄位以 pandas 讀取後在 Python 端組成列聯表或兩組數列;本模組不另開 SQL 檔,理由見決策日誌 [M3-1]。

**與 M2 的銜接**:檢定對象與分組界線全部沿用 M2 的定義:H4 沿用階段 C 的淺關係旗標(total_relationship_count ≤ 2),H6 沿用階段 B 的逐值分組(0–6 次),H8 沿用性別欄位,total_trans_amt 沿用階段 E 對其右偏形狀的確認。承接的待驗證事項:#12(事後發現二則是否進檢定)、#15(H6 因果方向)、#16(H8 統計顯著 vs 實務顯著)。

**判讀格式**:每個檢定固定四段:實測(統計量、p 值、效果量)/ 判定(校正後是否統計顯著、是否實務顯著)/ 對決策的意涵 / 待辦或限制。

**執行原則**:本 notebook commit 前一律 Restart & Run All,以確保由上而下執行必定重現相同結果。

Cell M3-2(markdown)|檢定範圍:檢定什麼、不檢定什麼、為什麼

## 第 1 節|檢定範圍與四個檢定對象

### 1.1 檢定的候選池是事前固定的

檢定次數越多,純靠抽樣波動得到「顯著」結果的機會就越多(說明見第 2 節 2.4)。因此,決定「檢定什麼」的方式,本身就是檢定結論可信度的一部分。本專案的做法是:候選池在 M2 切片之前就固定為假設樹的九條假設(H1–H9,見 notebook 02 Cell 2 與決策日誌 [M2-1]),每條假設寫明對應欄位與預期方向,且假設樹的 commit 時間早於任何切片結果。本模組的四個檢定對象,三個直接來自這九條假設(H6、H4、H8),第四個(total_trans_amt)是 notebook 02 階段 E 在進入本模組前已預告的檢定對象(Cell E-1)。

沒有進入候選池的欄位一律不檢定:次要檢視清單的六個欄位(教育程度、婚姻狀態、扶養人數、卡別、年齡、信用額度)與兩則事後發現(「Doctorate」流失率偏高、「收入 Unknown × 低額度」流失率偏高)。理由是:這些觀察是看過切片結果之後才出現的,如果再用同一份資料對它們做檢定,等於用「產生假設的那份資料」去「驗證同一個假設」,屬循環論證;依 [M2-1] 的流程,它們只能列為觀察,等取得新資料後再驗證。待驗證事項 #12 於此結案(結論:不進檢定)。

### 1.2 從候選池中挑出四個對象的標準

候選池內有九條假設,本模組只檢定其中三條加一個連續變數。挑選標準不是切片結果的顯著程度,而是以下三項:

1. **決策相關性**:檢定結果會直接影響某個後續模組的決策。H4 影響 M5 的名單條件與「深化往來」策略;H6 影響 M5 的預警規則;total_trans_amt 影響 M5 價值軸的正當性。
2. **示範價值**:H8(性別)在 M2 已判定「有關聯但組間差距小」,它是本專案用來示範「統計顯著不等於實務顯著」的案例,這個區分是分析師判斷力的核心,值得用一個檢定完整呈現。
3. **方法覆蓋**:前三個對象都是「類別分組 × 流失與否」的比例比較;補一個連續變數(total_trans_amt)的兩組比較,使本模組涵蓋流失分析會用到的三類檢定方法:多類別關聯(卡方獨立性檢定)、兩組比例比較(兩比例 z 檢定)、兩組連續數值比較且資料右偏(Mann-Whitney U 檢定)。

候選洞察清單 v2 中訊號最強的幾條(低活躍、零動用、命中數分層、極高風險組合客群)反而不做檢定,理由有二:第一,這些客群的差距幅度(diff_pp +20 以上、組合格達 +50 至 +79)與樣本規模(數百至數千人)使檢定結果毫無懸念,對它們做檢定不會改變任何決策,只會多消耗一個檢定名額(見 2.4 的多重比較成本);第二,它們真正待回答的問題不是「差距是否為真」,而是「訊號之間的交互作用」(notebook 02 Cell C-6),此問題應由 M4 的多變數模型以交互項處理,不是兩兩檢定能回答的。(決策日誌 [M3-0])。

### 1.3 四個檢定對象的來歷、目的與若不顯著的後果

| 檢定 | 對象 | 來歷 | 支持的決策 | 若不顯著會影響什麼 |
|---|---|---|---|---|
| T1 | H6 客服聯繫次數(0–6 次)× 是否流失 | 假設樹枝 3「服務摩擦」;M2 判定強支持(1.75% → 100% 完整單調);候選洞察清單 v2 主力層 #5 | 決策 2:聯繫次數能否作為高風險客戶的可辨識訊號;M5 預警名單規則(待驗證 #10-a)的前提 | 「高聯繫 ≥4 次」不得作為預警規則,M5 少一條名單條件 |
| T2 | H4 往來產品數(≤2 項 vs ≥3 項)× 是否流失 | 假設樹枝 2「關係深度」;M2 判定強支持(1–2 項 26–28%、3 項起 ≤17%);清單 v2 中段層 #6;分組沿用階段 C 淺關係旗標 | 決策 1:淺關係旗標作為 M5 名單條件的正當性;本專案規劃的 M5「深化區」策略(交叉銷售)的前提是「產品數多者流失率低」 | 淺關係不得作為名單條件;深化區策略失去資料依據,須重新設計 |
| T3 | H8 性別(女性 vs 男性)× 是否流失 | 假設樹枝 4「人口與產品屬性」;M2 判定有關聯但組間差距僅 2.74 個百分點;清單 v2 觀察層 #10 | 示範「統計顯著 vs 實務顯著」;結論預期為「不設計性別導向的挽留方案」,並作為 M7 公平性檢視的伏筆 | 不影響任何名單;示範案例改由其他變數負責 |
| T4 | total_trans_amt(近 12 個月交易金額)流失組 vs 留存組 | 不在候選洞察清單(它不是「哪個客群流失率高」的洞察);M5 依問題定義書決策 1 與本專案的分析規劃,以交易金額作為客戶價值的代理變數(資料集無利潤、手續費等直接價值欄位);階段 E Cell E-7 已確認右偏並預告為本模組對象 | 決策 1 的價值軸:確認流失組與留存組在此變數上確實不同,交易金額能區分客群,可作為 M5 風險 × 價值矩陣的價值軸 | 價值軸須改用其他變數(如循環餘額),M5 矩陣設計要調整 |

### 1.4 兩層防線:防「檢定對象怎麼來」與防「四次檢定的累積風險」

本模組對多重比較問題(2.4)採兩層處理,兩層防的問題不同,不能互相取代:

- **第一層是流程**:候選池在切片前固定(1.1),檢定對象依決策相關性從候選池挑選(1.2),不依切片結果的顯著程度挑選。這一層防的是「從上百種可能的比較裡挑出最顯著的贏家來檢定」:若如此做,挑到的很可能是純靠抽樣波動最顯著的比較,再用同一份資料檢定只會把這個波動再確認一次,檢定失去意義。
- **第二層是 Bonferroni 校正**:即使只做四次檢定,四次合計的假陽性機率仍高於單次的 5%;本模組把顯著水準 α 從 0.05 調整為 0.05 ÷ 4 = 0.0125,每個檢定的 p 值須小於 0.0125 才判定為統計顯著。這一層防的是「四次檢定本身的累積風險」。校正方法的選擇與取捨見 2.4 與決策日誌 [M3-2]。

**更嚴格的對照**:若把 Bonferroni 的除數取為候選池全部的九條假設(α = 0.05 ÷ 9 ≈ 0.0056),本模組四個檢定的 p 值仍全部遠低於此門檻(見各檢定實測),結論不變。此對照列出的目的是揭露「挑選標準即使被質疑為偏向強訊號,也不影響任何結論」。

Cell M3-3(markdown)|統計概念說明

## 第 2 節|本模組使用的統計概念

以下四段各以「定義 + 本案數字的例子」說明,是判讀各檢定時的共同依據。

### 2.1 虛無假設、對立假設、p 值

**虛無假設(null hypothesis, H0)**:兩組的真實流失率相同。**對立假設(alternative hypothesis, H1)**:兩組的真實流失率不同。檢定的邏輯永遠是先假設 H0 為真,再計算「在 H0 為真的情況下,看到目前這份資料(或差距更大的資料)的機率」,這個機率就是 **p 值(p-value)**。檢定只能「拒絕 H0」或「無法拒絕 H0」,不能證明 H0 為真:找不到差距的證據,不等於證明沒有差距。

為什麼全體 10,127 名客戶都在資料裡,仍然要談「抽樣」:分析的對象不是這一萬人此刻的狀態,而是這家銀行客戶流失的規律;同一套規律下,下一批客戶的數字會不同。這一萬人是該規律的一次觀察,檢定回答的是「換一批客戶,這個差距還會存在嗎」。

**p 值的計算邏輯**:即使兩組的真實流失率相同,任何一次抽樣算出的兩組差距也不會恰好是 0,而是在 0 附近隨機波動;波動的典型幅度可由流失率與兩組人數算出,稱為**標準誤(standard error)**。把觀察到的差距除以標準誤,得到「這個差距是典型波動的幾倍」,即**檢定統計量**(兩比例檢定為 z 值);統計量越大,代表差距越難以用隨機波動解釋,p 值越小。z 值與 p 值的對應是固定的:z = 2 對應 p ≈ 0.05,z = 2.5 對應 p ≈ 0.0125,z = 3.8 對應 p ≈ 0.0001。

**本案數字**:H8 女性流失率 17.36%、男性 14.62%,差距 2.74 個百分點;兩組差距的標準誤為 0.73 個百分點;z = 2.74 ÷ 0.73 ≈ 3.75,對應 p ≈ 0.0002。意思是:若男女的真實流失率相同,大約每五千次抽樣才會有一次出現 2.74 個百分點以上的差距,因此不接受「相同」這個解釋。

**p 值的業務語言**:p 值很小(例如 0.0002)代表「如果兩群人真的沒有差別,光靠抽樣波動看到這麼大差距的機會極低,所以這個差距不是巧合,可以作為決策依據」;p 值不小(例如 0.3)代表「如果兩群人真的沒有差別,每十次抽樣也有三次會看到這種差距,無法排除只是抽樣波動,目前證據不足以據此做決定」。但這不是證明兩群人沒有差別。

**p 值的三個常見誤解**:(1) p = 0.03 不是「有 97% 的機率兩組有差」:p 值是「H0 為真時看到此資料的機率」,不是「H0 為真的機率」;(2) p 值不代表差距大小:p 值小可能是差距大,也可能只是樣本大,本案 H8 即為後者,「差距多大」是效果量的工作;(3) p 值大不代表證明沒差:可能是樣本不夠大而看不出來(見 2.5 檢定力)。

**顯著水準 α**:事前訂定的門檻,p 值小於 α 即拒絕 H0。α 代表我們願意承擔的假陽性風險(H0 其實為真、卻被誤判為有差,稱為型一錯誤 Type I error);相反的錯誤(H1 其實為真、卻未被檢出,稱為型二錯誤 Type II error)由檢定力控制。本專案採 α = 0.05,此為統計學與商業分析的通行預設值(A/B 測試平台預設亦為 0.05),假陽性代價特別高的情境(如藥物核准)才會改用 0.01。α 必須在計算 p 值之前訂定,看到 p 值之後再調整門檻等同為了配合結論而更改規則。本模組因多重比較校正,實際門檻為 0.0125(見 2.4)。

### 2.2 效果量與信賴區間

**效果量(effect size)** 回答「差距有多大」,這是 p 值不回答的問題。不同資料型態用不同的效果量:

- 兩組比例比較:**比例差(percentage-point difference)** 配 **95% 信賴區間(confidence interval, CI)**。信賴區間的意思是:把抽樣波動考慮進去後,真實差距有 95% 的把握落在此範圍內。「信賴區間不包含 0」與「p < 0.05」在決策上等價(兩者同時成立、同時不成立),但信賴區間額外提供差距的範圍:本案 H8 的差距為 2.74 個百分點,信賴區間 1.32 至 4.17,代表真實差距最小可能只有 1.3、最大可能到 4.2。範圍的實務用途是 M5 估算預期效益時可用下限做保守估計、上限做樂觀估計。同時報告**勝算比(odds ratio)**:勝算 = 流失人數 ÷ 未流失人數,勝算比 = 兩組勝算相除,表示「A 組流失的勝算是 B 組的幾倍」;M4 的 logistic regression 以同一語言解讀係數,本模組先行建立。
- 多類別關聯(卡方檢定):**Cramér's V**,0 至 1,越大代表兩變數關聯越強。
- 兩組連續數值比較(Mann-Whitney):**rank-biserial correlation**,−1 至 1,並附兩組中位數作為業務語言的對照。

效果量大小的慣例(Cohen 提出,社群通用):Cramér's V 與 rank-biserial 皆以 0.1 為弱、0.3 為中、0.5 為強;此慣例描述的是「變數把兩組分開的程度」。

**統計顯著與實務顯著**:
統計顯著 = p 值小於門檻,差距不是抽樣波動;實務顯著 = 差距大到值得為它採取行動。兩者獨立,可以只滿足其一。<br/>

實務顯著沒有通用的數字門檻,它取決於行動的成本、行動的效益、以及差距相對於基準的大小;本資料集沒有挽留成本與客戶利潤資料,無法精算。
本模組採用的工作判準(判斷是否為實務顯著):
1. 先看該檢定所比較的那兩個數字的差距。
例如:兩比例 z 檢定比的是兩個組別(女 vs 男、淺 vs 非淺),所以判準看「組間差距」。卡方檢定比的是七個組與「無關聯時的期望」,沒有單一的「兩組之差」可看;所以若要回答「聯繫幾次以上才值得行動」,就要回到 M2 候選洞察清單的判準,看「每一組對基準的 diff_pp」。這也是 [M2-10] 分層門檻原本的定義(主力層 ≥ +15、中段層 ≥ +9 都是 diff_pp)。
2. 該檢定所比較的那兩個數字之差若達 +9 個百分點以上,視為「實務顯著」(值得考慮行動):兩組比較(T2、T3)看組間差距,多組比較(T1)看各組對基準 16.07% 的 diff_pp; +9 沿用候選洞察清單 v2 中段層的門檻([M2-10],該門檻以 diff_pp 定義)。未達者視為「統計上有差,但不值得單獨為它設計方案」;此判準為本案自訂,最終是否行動由 M5 結合成本效益定案。

### 2.3 檢定的前提條件

每一種檢定都有它的數學依據成立的前提,前提不成立時算出的 p 值不可信。本模組使用的三種檢定的前提如下,各檢定的方法段會逐一檢查:

- 兩比例 z 檢定:(1)每一組的流失人數與未流失人數都不能太少(慣例各至少 10 人),因為 p 值的計算依賴「差距的隨機波動呈鐘形分布」的近似,人數太少時近似不準;(2)各觀察值彼此獨立(一位客戶流失與否不影響另一位),本案一列一客戶(稽核 [A]),成立。
- 卡方獨立性檢定:每一格的期望人數(無關聯時該格應有的人數)不宜低於 5,原因同上;不滿足時將相鄰小組合併後再檢定。
- Mann-Whitney U 檢定:不假設資料的分布形狀,前提只有觀察值獨立;這正是它適用於右偏資料的原因。

### 2.4 多重比較與 Bonferroni 校正

**問題**:α = 0.05 代表單一檢定有 5% 的機率把不存在的差異誤判為存在。若做 k 次檢定,至少一次誤判的機率為 1 − 0.95^k:四次是 18.5%,二十次是 64%。檢定次數越多,得到假發現的機率越高,這稱為**多重比較問題(multiple comparisons problem)**。M2 的 [M2-1] 以事前指定假設樹限制切片數量,防的就是同一個問題。

**Bonferroni 校正**:把 α 除以檢定次數,作為每個檢定的新門檻。本模組四個檢定,α = 0.05 ÷ 4 = 0.0125。校正後,四個檢定「至少一次假陽性」的合計機率被壓回 5% 以下。

**取捨**:門檻變嚴後,假陽性減少,但假陰性(真的有差卻未被檢出)增加,因為 p 值必須更小才能通過。在「真實差距不大」的情境下,這可能錯過真訊號:例如某差距的 p 值為 0.02,不校正時顯著,校正後(門檻 0.0125)不顯著。統計上有更不容易錯過真訊號的校正法(Holm 法、控制錯誤發現率的 FDR 法),它們在「檢定數多且差距小」的情境優於 Bonferroni。

**本案採用 Bonferroni 的理由**:四個檢定對象的差距都很大(H6 各組流失率自 1.75% 至 100%,H4 差 13.75 個百分點,total_trans_amt 中位數相差 1.8 倍),p 值預期都遠低於 0.001,門檻是 0.05 或 0.0125 不會改變任何一個結論:門檻變嚴在本案幾乎沒有代價,卻換到「四個結論的合計假陽性率被壓回 5% 以內」與「對非技術專業決策者,用一句話即可說明此方法」。更精細的校正法在本案不會改變結論,且對業務單位解釋的成本較高,故不採用。(決策日誌 [M3-2])


### 2.5 檢定力

**定義**:**檢定力(statistical power)** = 若真實差距存在,此次檢定能檢出它(p 值小於門檻)的機率,等於 1 減去型二錯誤的機率。慣例目標為 0.8,來源是 Cohen(1988)的建議,之後成為實驗設計與 A/B 測試的預設值;它與 α = 0.05 的關係是假陰性容忍 20%、假陽性容忍 5%,反映「把不存在的差異當真」通常比「漏掉一個真差異」代價更高。

**檢定力由什麼決定**:樣本數越大、真實差距越大,檢定力越高。原因是 p 值取決於「差距是標準誤的幾倍」,樣本越大標準誤越小,差距越大倍數越大,p 值越容易落在門檻內。本案 H4 的真實差距約 13.75 個百分點、標準誤 0.89 個百分點,差距是標準誤的 15 倍以上,要讓 p 值不顯著,抽樣波動必須把差距壓到 2.5 倍標準誤(約 2.2 個百分點)以下,發生機率趨近於零,檢定力為 1.0000;H8 的差距 2.74、標準誤 0.73、倍數 3.75,檢定力在 α = 0.0125 下為 0.90。因此本模組四個檢定都不會發生「真的有差卻測不出來」的情況。

**實務意義**:當 p 值不顯著時,必須先問「是真的沒差,還是樣本不夠大而看不出來」;檢定力低的檢定,不顯著不能作為結論。本案樣本大、差距大,檢定力不構成風險;但檢定力在 M6 設計 A/B 測試時是核心變數,所以 M6 實驗前必須計算「要檢出預期的效果,每組需要多少人」。第 8 節以 H8 的實測效果量示範此計算,作為 M6 的銜接點。

In [1]:
# Cell M3-4(code)|環境設定與資料載入

import sys
import pathlib
sys.path.append(str(pathlib.Path.cwd().parent))  # 將專案根目錄加入模組搜尋路徑,才能 import src/db.py

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep, proportion_effectsize
from statsmodels.stats.power import NormalIndPower
from src.db import get_engine

engine = get_engine()

# 本模組共用常數
BASELINE_CHURN_PCT = 16.07   # 全案基準流失率(問題定義書 §3)
ALPHA = 0.05                 # 事前訂定的顯著水準(2.1)
N_TESTS = 4                  # 本模組檢定次數(T1–T4)
ALPHA_ADJ = ALPHA / N_TESTS  # Bonferroni 校正後的門檻 = 0.0125(2.4、[M3-2])

# 一次讀取四個檢定需要的欄位:每列一位客戶,共 10,127 列。
# 檢定的輸入(列聯表、兩組數列)在 Python 端由此 DataFrame 組成,不另開 SQL 檔([M3-1])。
# 「兩組數列」指 T4 的輸入:1,627 位流失客戶與 8,500 位留存客戶各自的交易金額數值序列。Mann-Whitney 的輸入就是這兩組數列。
df = pd.read_sql("""
    SELECT clientnum, is_churned,
           contacts_count_12_mon,        -- T1:H6 聯繫次數
           total_relationship_count,     -- T2:H4 往來產品數
           gender,                       -- T3:H8 性別
           total_trans_amt               -- T4:近 12 個月交易金額
    FROM clean.customers
""", engine)

# 讀取後的一致性核對:筆數與基準流失率必須等於 M1 定案的數字,否則後續所有檢定的分母都不對
assert len(df) == 10_127, f"筆數 {len(df)} != 10,127"
assert round(df["is_churned"].mean() * 100, 2) == BASELINE_CHURN_PCT, "基準流失率與問題定義書不符"
print(f"載入 {len(df):,} 列 | 基準流失率 {df['is_churned'].mean():.2%} | 校正後門檻 α = {ALPHA_ADJ}")

載入 10,127 列 | 基準流失率 16.07% | 校正後門檻 α = 0.0125


In [2]:
# Cell M3-5(code)|兩比例 z 檢定的共用函式(T2、T3 共用)

# 功能:給定一個布林遮罩(mask,True 為 A 組、False 為 B 組),對 A、B 兩組的流失率做兩比例 z 檢定,
#       並回傳判讀所需的全部數字:兩組人數與流失率、合併比例、標準誤、z 值、p 值、比例差與 95% 信賴區間、勝算比與其信賴區間、Cohen's h。
# 作法(對應第 2 節 2.1 的計算邏輯):
#   1. 分別數出兩組的人數(n)與流失人數(churn)。
#   2. 在 H0(兩組流失率相同)下,兩組共用同一個流失率,其最佳估計是把兩組合併後的總流失率(pooled)。
#   3. 用 pooled 與兩組人數算出「兩組差距純靠抽樣波動的典型幅度」(標準誤 se)。
#   4. z = 觀察到的差距 ÷ se;proportions_ztest 內部即依此計算並回傳 z 與雙尾 p 值。
#   5. 比例差的 95% 信賴區間以 Wald 法計算(confint_proportions_2indep,method="wald"),本案每組流失與未流失人數皆數百以上,Wald 法適用。
#   6. 勝算比 = (A 組流失 ÷ A 組未流失) ÷ (B 組流失 ÷ B 組未流失);其信賴區間在對數尺度上計算後再取指數,為勝算比的標準做法。
#   7. Cohen's h 為兩比例差距的標準化效果量(與人數無關),供第 8 節檢定力計算使用。
# 前提檢查:每組流失人數與未流失人數皆須 ≥ 10(2.3),函式內以 assert 把關。

def two_proportion_test(mask: pd.Series, label_a: str, label_b: str) -> dict:
    a = df.loc[mask, "is_churned"]
    b = df.loc[~mask, "is_churned"]
    churn_a, n_a = int(a.sum()), len(a)
    churn_b, n_b = int(b.sum()), len(b)
    for name, c, n in [(label_a, churn_a, n_a), (label_b, churn_b, n_b)]:
        assert min(c, n - c) >= 10, f"{name} 的流失或未流失人數不足 10,兩比例 z 檢定前提不成立"

    p_a, p_b = churn_a / n_a, churn_b / n_b
    pooled = (churn_a + churn_b) / (n_a + n_b)
    se = np.sqrt(pooled * (1 - pooled) * (1 / n_a + 1 / n_b))
    z, p = proportions_ztest([churn_a, churn_b], [n_a, n_b])
    ci_lo, ci_hi = confint_proportions_2indep(churn_a, n_a, churn_b, n_b, method="wald")

    odds_ratio = (churn_a / (n_a - churn_a)) / (churn_b / (n_b - churn_b))
    se_log_or = np.sqrt(1 / churn_a + 1 / (n_a - churn_a) + 1 / churn_b + 1 / (n_b - churn_b))
    or_lo, or_hi = np.exp(np.log(odds_ratio) - 1.96 * se_log_or), np.exp(np.log(odds_ratio) + 1.96 * se_log_or)

    return {
        "group_a": label_a, "n_a": n_a, "churn_a": churn_a, "rate_a_pct": round(100 * p_a, 2),
        "group_b": label_b, "n_b": n_b, "churn_b": churn_b, "rate_b_pct": round(100 * p_b, 2),
        "pooled_pct": round(100 * pooled, 2), "se_pp": round(100 * se, 3),
        "diff_pp": round(100 * (p_a - p_b), 2),
        "ci95_pp": (round(float(100 * ci_lo), 2), round(float(100 * ci_hi), 2)),
        "z": round(float(z), 2), "p_value": float(p),
        "significant_adj": bool(p < ALPHA_ADJ),
        "odds_ratio": round(float(odds_ratio), 3), "or_ci95": (round(float(or_lo), 3), round(float(or_hi), 3)),
        "cohen_h": round(float(proportion_effectsize(p_a, p_b)), 3),        # 呈現用(四捨五入)
        "cohen_h_raw": float(proportion_effectsize(p_a, p_b)),              # 計算用(不四捨五入),供第 8 節檢定力計算
    }

def show_result(res: dict) -> None:
    """把檢定結果印成判讀用的摘要。"""
    print(f"{res['group_a']}: n={res['n_a']:,}, 流失 {res['churn_a']:,} 人, 流失率 {res['rate_a_pct']}%")
    print(f"{res['group_b']}: n={res['n_b']:,}, 流失 {res['churn_b']:,} 人, 流失率 {res['rate_b_pct']}%")
    print(f"合併流失率 {res['pooled_pct']}% | 差距的標準誤 {res['se_pp']} pp")
    print(f"比例差 {res['diff_pp']} pp, 95% CI {res['ci95_pp']}")
    print(f"z = {res['z']}, p = {res['p_value']:.3g}, 校正後門檻 {ALPHA_ADJ} → 統計顯著: {res['significant_adj']}")
    print(f"勝算比 {res['odds_ratio']}, 95% CI {res['or_ci95']} | Cohen's h = {res['cohen_h']}")

Cell M3-6(markdown)|T2 方法段:H4 往來產品數

## T2|H4 往來產品數 × 是否流失:兩比例 z 檢定

**假設**:
- H0:淺關係客戶(往來產品 ≤2 項)與非淺關係客戶(≥3 項)的真實流失率相同。
- H1:兩組的真實流失率不同。

**分組界線的來源**:≤2 項 vs ≥3 項沿用 notebook 02 階段 C 的淺關係旗標(Cell C-1、決策日誌 [M2-7]),界線依據是階段 B 逐值分組的實測斷點:1 項 25.60%、2 項 27.84%、3 項 17.35%、4 項以上 10.50–12.00%,流失率在 2 項與 3 項之間下降約 10 個百分點(notebook 02 Cell 18)。檢定沿用旗標分組而不另訂界線,理由是 M5 名單的圈選條件就是這條旗標;檢定的分組若與名單條件不同,檢定通過也無法直接支持名單(決策日誌 [M3-1])。

**為什麼選兩比例 z 檢定**:方法跟著資料型態走。分組變數是兩類(淺關係 / 非淺關係),結果變數是兩類(流失 / 留存),要比較的是兩組的流失率(比例)是否不同,對應的標準方法就是兩比例 z 檢定。

**檢定在做什麼**:先在 H0 下假設兩組共用同一個流失率(以合併後的總流失率 16.07% 估計),據此算出兩組差距純靠抽樣波動的典型幅度(標準誤);再把觀察到的差距除以標準誤得到 z 值,即「觀察到的差距是典型波動的幾倍」;z 值越大,差距越難以用波動解釋,p 值越小。

**前提檢查**:淺關係組 2,153 人(流失 579、未流失 1,574),非淺關係組 7,974 人(流失 1,048、未流失 6,926),每組的流失與未流失人數都遠超過 10 人;一列一客戶,觀察值獨立。前提成立。

**同時報告的效果量**:比例差與 95% 信賴區間(差距的大小與範圍)、勝算比(供 M4 銜接)。

In [3]:
# Cell M3-7(code)|T2 執行:H4 淺關係(≤2 項)vs 非淺關係(≥3 項)

res_t2 = two_proportion_test(df["total_relationship_count"] <= 2, "淺關係(≤2 項)", "非淺關係(≥3 項)")
show_result(res_t2)


淺關係(≤2 項): n=2,153, 流失 579 人, 流失率 26.89%
非淺關係(≥3 項): n=7,974, 流失 1,048 人, 流失率 13.14%
合併流失率 16.07% | 差距的標準誤 0.892 pp
比例差 13.75 pp, 95% CI (11.74, 15.76)
z = 15.42, p = 1.26e-53, 校正後門檻 0.0125 → 統計顯著: True
勝算比 2.431, 95% CI (2.166, 2.728) | Cohen's h = 0.348


Cell M3-8(markdown)|T2 判讀

**T2 判讀|H4 往來產品數**

- **實測**:淺關係客戶(≤2 項,n=2,153)流失率 26.89%,非淺關係客戶(≥3 項,n=7,974)流失率 13.14%,比例差 13.75 個百分點,95% 信賴區間 11.74 至 15.76 個百分點;z = 15.42,p ≈ 10⁻⁵³;勝算比 2.43(95% CI 2.17 至 2.73),即淺關係客戶流失的勝算是非淺關係客戶的 2.4 倍。
- **判定**:統計顯著(p 遠低於校正後門檻 0.0125,亦遠低於更嚴格的 α/9 ≈ 0.0056);實務顯著(差距 13.75 個百分點,信賴區間下限 11.74 仍高於本模組 +9 個百分點的工作判準)。z 值達 15.42,表示觀察到的差距是抽樣波動典型幅度的 15 倍以上;若兩組真實流失率相同,不可能靠抽樣得到這樣的差距。H4「往來產品數越少,流失率越高」的假設成立。
- **對決策的意涵**:(1) 淺關係旗標(≤2 項)作為 M5 名單條件的前提成立:它圈出的客群流失率確實高於其餘客戶,且差距大到值得行動;M5 估算對此客群投放挽留的預期效益時,可以信賴區間下限 11.74 個百分點做保守估計、上限 15.76 做樂觀估計。(2) 本專案規劃的「深化區」策略(以交叉銷售提高往來產品數)所依賴的關聯:「產品數多者流失率低」在資料上成立;但本檢定證明的是關聯,不是「增加產品數會降低流失」的因果,深化策略是否有效須由 M6 實驗驗證。(3) 往來產品數列入 M4 特徵,其與流失的關聯已有推論性證據。
- **待辦或限制**:本檢定比較的是全體淺關係客戶與全體非淺關係客戶,是把低活躍與活躍正常的客戶混在一起計算的平均差距。notebook 02 Cell C-6 的實測顯示,淺關係在活躍正常子群內只使流失率提高 4.0 個百分點(2.66% → 6.66%),在低活躍子群內則提高 60.5 個百分點(25.76% → 86.29%)。因此本檢定通過只能支持「整體而言淺關係與流失有關」,不能推論「對每一位淺關係客戶投放挽留都划算」:對活躍正常的淺關係客戶(流失率 6.66%,低於基準 16.07%)投放是浪費。M5 的名單條件應為「淺關係且低活躍」,而非「淺關係」單獨成立(決策日誌 [M2-8])。

Cell M3-9(markdown)|T3 方法段:H8 性別

## T3|H8 性別 × 是否流失:兩比例 z 檢定(統計顯著與實務顯著的示範案例)

**假設**:
- H0:女性與男性客戶的真實流失率相同。
- H1:兩組的真實流失率不同。

**本檢定的角色**:H8 在 M2 判定「有關聯,但組間差距僅 2.74 個百分點」(notebook 02 Cell 9),列入候選洞察清單 v2 的觀察層,預定作為「統計顯著 vs 實務顯著」的示範(待驗證事項 #16)。它的價值不在於支持任何名單,而在於完整呈現「檢定通過不等於該採取行動」的判斷:兩組樣本都接近五千人,即使差距很小,p 值也會很小;判讀時若只看 p 值,會誤以為性別是值得行動的訊號。

**為什麼選兩比例 z 檢定**:分組兩類(女性 / 男性)、結果兩類(流失 / 留存),比較兩組比例,與 T2 相同。

**前提檢查**:(1)女性 5,358 人(流失 930、未流失 4,428),男性 4,769 人(流失 697、未流失 4,072),每組流失與未流失人數皆遠超過 10 人;(2)觀察值獨立。前提成立。

In [4]:
# Cell M3-10(code)|T3 執行:H8 女性 vs 男性

res_t3 = two_proportion_test(df["gender"] == "F", "女性", "男性")
show_result(res_t3)

女性: n=5,358, 流失 930 人, 流失率 17.36%
男性: n=4,769, 流失 697 人, 流失率 14.62%
合併流失率 16.07% | 差距的標準誤 0.731 pp
比例差 2.74 pp, 95% CI (1.32, 4.17)
z = 3.75, p = 0.000176, 校正後門檻 0.0125 → 統計顯著: True
勝算比 1.227, 95% CI (1.102, 1.366) | Cohen's h = 0.075


Cell M3-11(markdown)|T3 判讀

**T3 判讀|H8 性別**

- **實測**:女性(n=5,358)流失率 17.36%,男性(n=4,769)流失率 14.62%,比例差 2.74 個百分點,95% 信賴區間 1.32 至 4.17 個百分點;z = 3.75,p = 0.00018;勝算比 1.23(95% CI 1.10 至 1.37);Cohen's h = 0.075。<br/>

- **判定**:統計顯著(p = 0.00018,低於校正後門檻 0.0125,亦低於 α/9);**實務不顯著**(差距 2.74 個百分點,信賴區間上限 4.17 仍遠低於 +9 個百分點的工作判準;Cohen's h = 0.075 低於「弱效果」的慣例門檻0.2)。兩個判定同時成立且方向相反,這正是本檢定要示範的情況。<br/>

- **對決策的意涵**:(1) 「性別與流失有關聯」在資料上為真(若男女真實流失率相同,約每五千次抽樣才會有一次(p=0.00018)出現 2.74 個百分點以上的差距),但差距太小,不值得為此設計性別導向的挽留方案:即使對全部女性客戶投放,名單流失率 17.36% 與基準 16.07% 幾乎沒有差別,挽留資源會被大量投放在不會流失的客戶上。(2) 由此案例得到的判讀原則:p 值只回答「差距是否為真」,不回答「差距是否夠大」;樣本越大,越小的差距也會統計顯著,判讀必須同時看效果量與其對基準的相對大小。(3) 性別不作為任何名單或策略的依據;M7 公平性檢視時,此結果同時是「模型風險分數若在性別間有差異,差異是否來自真實流失率差異」的比較基準。<br/>

- **待辦或限制**:本檢定只證明流失率在性別間有微小差異,不涉及原因;性別與其他行為變數的關係(例如女性是否因產品持有結構不同而流失率略高)不在本模組範圍,若 M4 模型顯示性別係數顯著,再回頭對照此結果。待驗證事項 #16 於此結案。

Cell M3-12(markdown)|T1 方法段:H6 客服聯繫次數

## T1|H6 客服聯繫次數 × 是否流失:卡方獨立性檢定

**假設**:
- H0:客服聯繫次數與是否流失兩個變數相互獨立,即知道客戶近 12 個月聯繫幾次,對判斷他是否流失沒有幫助;各聯繫次數組的真實流失率相同。
- H1:兩個變數有關聯,即各聯繫次數組的真實流失率不全相同。

**分組的來源**:聯繫次數為 0 至 6 次的整數,沿用 notebook 02 階段 B 對低基數整數欄的逐值分組(Cell 20、21),七組各自為一列。不把七組合併成「≥4 次 vs <4 次」兩組再用兩比例檢定,理由是 H6 的證據價值在整條曲線:M2 判定強支持的依據是七個點完整單調上升(1.75% → 7.20% → 12.49% → 20.15% → 22.63% → 33.52% → 100%),合併後此資訊消失;而且合併的界線(≥4 次)是 M5 尚未定案的預警線(待驗證事項 #10-a),檢定不宜預先綁定它。(決策日誌 [M3-1])

**為什麼選卡方獨立性檢定**:方法跟著資料型態走。分組變數有七類,結果變數有兩類,要回答的是「分組變數與結果變數有沒有關聯」,對應的標準方法是卡方獨立性檢定(chi-square test of independence)。它適用於任何「多類別 × 多類別」的列聯表。

**檢定在做什麼**:
1. 把資料排成列聯表(contingency table):7 列(聯繫次數)× 2 欄(流失 / 留存),每格是人數。
2. 在 H0(無關聯)下,每一組的流失率都應該等於全體的 16.07%,所以每格「應該」有的人數 = 該列總人數 × 該欄總人數 ÷ 10,127,稱為期望人數(expected count)。例如 0 次組 399 人,無關聯時應有 399 × 16.07% ≈ 64 人流失。
3. 每格計算(實際人數 − 期望人數)² ÷ 期望人數,14 格加總得到卡方統計量 χ²。平方是讓正負落差都計入,除以期望人數是讓人數多的格與人數少的格可以用同一把尺比較;純靠抽樣波動時,每一格的貢獻平均約為 1。
4. 以 χ² 與自由度查 p 值。自由度(degrees of freedom)= (列數 − 1) × (欄數 − 1) = 6,意思是列總和與欄總和固定後,14 格中只有 6 格的人數可以自由變動,其餘由總和決定;自由度告訴查表程式「無關聯時 χ² 純靠波動平均會累積到多少」(約等於自由度,本案約 6),程式據此選用正確的參考分布計算 p 值。
5. p 值的意思是「若兩變數真的無關聯,14 格的落差純靠抽樣波動累積到這麼大 χ² 的機率」;p 值小於門檻即拒絕 H0。

**前提檢查**:每格期望人數不宜低於 5(第 2 節 2.3),因為期望人數過小的格子,一兩個人的隨機變動就會讓該格的貢獻值劇烈跳動,p 值會失真。本案人數最少的兩組為 5 次組(176 人)與 6 次組(54 人),其流失格的期望人數分別為 176 × 16.07% ≈ 28.3 與 54 × 16.07% ≈ 8.7,皆高於 5,14 格全部通過;檢查結果由下方程式碼實測列出。若未通過,處理原則是把相鄰的小組合併(例如 5 次與 6 次合併為「≥5 次」)後再檢定。

**同時報告的效果量**:Cramér's V,由 χ² 換算,範圍 0 至 1。χ² 本身會隨樣本數放大(人數越多,即使關聯很弱 χ² 也會很大),所以不能用 χ² 的大小判斷關聯強弱;Cramér's V 把樣本數與表格大小的影響除掉,才是可以跨變數比較的關聯強度。慣例:0.1 弱、0.3 中、0.5 強。

**本檢定的限制**:卡方檢定只判斷「有沒有關聯」,不理會七組的先後順序;把七組的順序打亂,χ² 與 p 值完全相同。因此「聯繫次數越多、流失率越高」這個方向與形狀,卡方檢定本身證明不了,它的證據來自 M2 切片的完整單調曲線。兩者合起來才是完整的論述:卡方檢定證明關聯存在且不是抽樣波動,切片曲線描述關聯的方向與形狀。統計上另有專門檢定「順序趨勢」的方法(例如 Cochran-Armitage 趨勢檢定),本模組不採用:M2 的曲線已足以描述趨勢,再做趨勢檢定不會改變任何決策。

In [5]:
# Cell M3-13(code)|T1 執行:H6 聯繫次數 × 是否流失,卡方獨立性檢定

# 1. 列聯表:7 列(聯繫次數 0–6)× 2 欄(留存 False / 流失 True),每格為人數
ct = pd.crosstab(df["contacts_count_12_mon"], df["is_churned"])
ct.columns = ["retained", "churned"]
ct["n"] = ct["retained"] + ct["churned"]
ct["churn_pct"] = (100 * ct["churned"] / ct["n"]).round(2)
print("列聯表(附各組流失率):")
display(ct)

# 2. 卡方獨立性檢定
#    correction=False:Yates 連續性校正只適用於 2×2 表,本表為 7×2,不套用。
chi2, p_t1, dof, expected = stats.chi2_contingency(ct[["retained", "churned"]], correction=False)

# 3. 前提檢查:每格期望人數(H0 下該格應有的人數)是否皆 ≥ 5
expected_df = pd.DataFrame(expected, index=ct.index, columns=["retained_exp", "churned_exp"]).round(1)
min_expected = expected.min()
assert min_expected >= 5, f"最小期望人數 {min_expected:.1f} < 5,須合併相鄰小組後再檢定"

# 4. 每格對 χ² 的貢獻:(實際 − 期望)² ÷ 期望;用來看關聯主要由哪幾組驅動
contrib = ((ct[["retained", "churned"]].values - expected) ** 2 / expected)
contrib_df = pd.DataFrame(contrib, index=ct.index, columns=["retained_contrib", "churned_contrib"]).round(1)

# 5. 效果量 Cramér's V = sqrt(χ² ÷ (n × (min(列數, 欄數) − 1)));本表 min(7, 2) − 1 = 1
n_total = len(df)
cramers_v = float(np.sqrt(chi2 / (n_total * (min(ct[["retained", "churned"]].shape) - 1))))

print("\n期望人數(H0 下各格應有人數):")
display(expected_df)
print(f"最小期望人數 = {min_expected:.1f}(前提 ≥ 5:通過)")
print("\n各格對 χ² 的貢獻:")
display(contrib_df)
print(f"\nχ² = {chi2:.1f}, 自由度 = {dof}, p = {p_t1:.3g}, 校正後門檻 {ALPHA_ADJ} → 統計顯著: {p_t1 < ALPHA_ADJ}")
print(f"Cramér's V = {cramers_v:.3f}")

列聯表(附各組流失率):


,retained,churned,n,churn_pct
contacts_count_12_mon,,,,
0,392,7,399,1.75
1,1391,108,1499,7.20
2,2824,403,3227,12.49
3,2699,681,3380,20.15
4,1077,315,1392,22.63
5,117,59,176,33.52
6,0,54,54,100.00



期望人數(H0 下各格應有人數):


,retained_exp,churned_exp
contacts_count_12_mon,,
0,334.9,64.1
1,1258.2,240.8
2,2708.6,518.4
3,2837.0,543.0
4,1168.4,223.6
5,147.7,28.3
6,45.3,8.7


最小期望人數 = 8.7(前提 ≥ 5:通過)

各格對 χ² 的貢獻:


,retained_contrib,churned_contrib
contacts_count_12_mon,,
0,9.7,50.9
1,14.0,73.3
2,4.9,25.7
3,6.7,35.1
4,7.1,37.3
5,6.4,33.4
6,45.3,236.8



χ² = 586.6, 自由度 = 6, p = 1.78e-123, 校正後門檻 0.0125 → 統計顯著: True
Cramér's V = 0.241


Cell M3-14(markdown)|T1 判讀

**T1 判讀|H6 客服聯繫次數**

- **實測**:<br/>
χ² = 586.6,自由度 6,p ≈ 10⁻¹²³;Cramér's V = 0.241。前提檢查通過:14 格期望人數最小者為 6 次組的流失格(8.7 人),高於 5。各格對 χ² 的貢獻顯示關聯由兩端驅動:6 次組(實際流失 54 人、期望 8.7 人)一格貢獻 236.8,占 χ² 的四成;0 次組(實際 7 人、期望 64.1 人)貢獻 50.9;中間各組的貢獻在 25 至 73 之間。<br/>

- **判定**:<br/>
統計顯著(p 遠低於校正後門檻 0.0125,亦遠低於 α/9 ≈ 0.0056)。自由度 6 的表在無關聯時 χ² 純靠抽樣波動平均只會累積到 6 左右、超過 22.5 的機率已低於千分之一,實測 586.6 只有一個合理解釋:各組流失率的差異是真的,不是這一萬名客戶剛好抽到。H6「客服聯繫次數與流失有關聯」的假設成立。<br/>
效果量 Cramér's V = 0.241,介於弱(0.1)與中(0.3)之間,偏向中等;此數字描述的是「聯繫次數這個變數整體上把流失與留存分開的程度」,它低於切片曲線兩端(1.75% 與 100%)給人的直覺印象,原因是全體客戶有 96% 集中在 1 至 4 次組,這四組的流失率介於 7% 至 23%,差異相對溫和,兩端的極端組人數少,對整體關聯強度的貢獻有限。<br/>
實務顯著的判定不用 V 值,而看切片曲線的組間差距:4 次組流失率 22.63%(diff_pp +6.56)未達 +9 個百分點的工作判準,5 次組 33.52%(+17.45)與 6 次組 100%(+83.93)則遠超過;即「高聯繫」訊號的實務顯著程度取決於預警線劃在哪一次,此為待驗證事項 #10-a 留給 M5 的問題。<br/>

- **對決策的意涵**:<br/>
(1) 聯繫次數可作為高風險客戶的可辨識訊號(問題定義書決策2):本檢定確認三件事:(a) 關聯真實存在(p ≈ 10⁻¹²³);(b) 關聯強度中偏弱(Cramér's V = 0.241);(c) 實務顯著程度取決於預警線劃在幾次的位置,由 M5 判定。M5 以聯繫次數作為預警名單規則有推論性證據。<br/>

(2) 本模組不決定「預警線的位置」:卡方檢定只證明「整體有關聯」,不指出「幾次以上才值得行動」;依切片數字,≥4 次的名單流失率 25.65%(notebook 02 Cell 21 註),≥5 次則超過 33% 但人數僅 230 人,門檻取決於挽留預算與單位成本,由 M5 定案(待驗證事項 #10-a)。<br/>
本 notebook 所稱「預警線」、「預警名單」,指的是「以聯繫次數圈選高風險名單的門檻與名單本身(風險標記的用法)」,但不主張聯繫發生在流失決定之前;後者見下方待辦或限制第(1)條「因果方向未定」。<br/>
聯繫次數可以當圈選出高風險名單的條件(因為T1已證明:≥5 次的客戶流失率超過 33%,表示已有證據顯示「這群人流失率確實比較高」)(這是指「可以做風險標記」),但圈選名單裡可能有相當比例的客戶已經決定流失(快照資料無法區分聯繫發生在流失決定之前或之後),所以這份名單的挽留成功率可能偏低。這件事情(可做挽留名單,但挽留名單的成功率可能偏低)要寫進 M5 的「策略風險」、M6 要用實驗驗證「對高聯繫客戶介入到底救得回多少」;M8 報告不能說「聯繫是流失的前兆,及早介入就能防止」(因為因果方向未定)。<br/>

(3) 聯繫次數列入 M4 特徵;各格貢獻表顯示 6 次組流失率為 100%(54 人全數流失),M4 建模時此組會是「完全分離」的來源之一,係數估計可能不穩定,列入待辦(待驗證事項 #19)。<br/>

- **待辦或限制**:<br/>
(1) 因果方向未定(待驗證事項 #15):本檢定證明聯繫次數與流失有關聯,證明不了聯繫發生在流失決定之前。快照資料無法區分「客戶因不滿而頻繁聯繫、之後流失」與「客戶已決定流失,為辦理註銷或爭議而聯繫」兩種情形,6 次組 100% 流失更像後者的痕跡。因此 H6 的定位是「風險標記」而非「提前預警」,M8 報告須明示此限制,不得寫成「聯繫次數增加是造成流失的原因」(決策日誌 [M2-5])。<br/>
(2) 卡方檢定不檢驗順序趨勢,「聯繫越多次、流失率越高」的趨勢由 M2 切片曲線描述,非本檢定的結論。<br/>
(3) 6 次組 100% 流失(n=54)對 M4 的完全分離問題,新增待驗證事項 #19 (見第 11 節(Cell M3-23))。<br/>

Cell M3-15(markdown)|T4 方法段:total_trans_amt

## T4|近 12 個月交易金額(total_trans_amt)流失組 vs 留存組:Mann-Whitney U 檢定

**假設**:
- H0:流失組與留存組的交易金額分布相同,即隨機抽一位流失客戶與一位留存客戶,流失客戶金額較低與較高的機率各為一半。
- H1:兩組分布不同,即其中一組整體偏低。

**本檢定的角色**:<br/>
total_trans_amt 不是「哪個客群流失率高」的洞察,它在本專案的用途是 M5 風險 × 價值矩陣的價值軸:資料集沒有利潤、手續費等直接的客戶價值欄位,問題定義書決策 1 與本專案的分析規劃「以交易金額作為客戶價值的代理變數(proxy variable)」。此設計隱含一個前提:流失組與留存組在交易金額上確實不同,此變數能區分客群。
本檢定確認此前提。<br/>
另一個角色是方法覆蓋:T1 至 T3 都是比例的檢定,本檢定補上「連續數值的兩組比較,且資料右偏」這一類的檢定。

**為什麼不用 t 檢定,而用 Mann-Whitney U 檢定**(決策日誌 [M3-3]):

1. t 檢定(t-test)是比較兩組「平均數」是否不同的方法,是連續數值兩組比較最常見的檢定;它的邏輯與兩比例 z 檢定相同(平均數差距 ÷ 差距的標準誤),前提是兩組資料的分布接近常態(鐘形),或樣本大到讓平均數的抽樣波動接近鐘形。<br/>
2. 本案不以 t 檢定為主要方法,原因有二。
第一,notebook 02 階段 E(Cell E-7、E-8)已確認 total_trans_amt 右偏:大多數客戶集中在低金額,少數高消費客戶把平均數拉高,流失組平均數 3,095 元、中位數僅 2,329 元,平均數高出中位數三成,代表一半的流失客戶消費在 2,329 元以下,平均數代表不了典型客戶。
第二,本檢定要回答的問題是「典型的流失客戶消費是否比典型的留存客戶低」,這是中位數層面的問題,不是平均數層面的問題;用平均數比較,結果會被少數高消費客戶主導。<br/>
3. 本案不採用 t 檢定的補充說明: 本案每組數千人,樣本大到 t 檢定在數學上仍然穩健(大樣本下平均數的抽樣波動會接近鐘形),所以本案不採用 t 檢定不是因為「不能用」,而是因為「回答的問題不對」。下方程式碼同時列出 t 檢定結果作為對照,兩者結論一致,但判讀以 Mann-Whitney 為準。


**Mann-Whitney U 檢定在做什麼**:
1. 把兩組全部 10,127 個金額混在一起,由小到大排名次(rank)。用名次而不用金額本身,是因為一位消費十萬元的客戶再極端也只占一個名次,不會像平均數那樣被拉走,這就是這個檢定不受右偏影響的原因。
2. 把每一位流失客戶(1,627 人)與每一位留存客戶(8,500 人)兩兩配對,共 1,627 × 8,500 = 13,829,500 對;每一對比較金額,流失客戶較高記 1 分,相同記 0.5 分。所有配對的分數加總即為 U 統計量(scipy 的 mannwhitneyu 以第一個傳入的組為準計算 U,本案第一組為流失組,故 U 數的是「流失客戶金額較高的配對數」)。
3. 若兩組分布相同,U 應接近總配對數的一半(約 6,914,750);U 值偏離「一半(這個數字)」越多,兩組差異越大。「U 與「一半(這個數字)」的差距」 除以「U 純靠抽樣波動的典型幅度」,得到 z 值,再換算成 p 值,邏輯與前三個檢定相同。

**前提檢查**:Mann-Whitney U 檢定不假設分布形狀,前提只有觀察值獨立;本案是一列一客戶,所以此前提成立。

**同時報告的效果量**:
- rank-biserial correlation:<br/>
先算「流失客戶金額較低的配對比例」P,兩組無差時 P = 0.5 ; rank-biserial = P − (1 − P) = 2P − 1, 範圍 −1 至 1。
rank-biserial = 0 代表兩組沒有差別 ; 1 代表所有流失客戶都比所有留存客戶消費低。
本 notebook 定義 rank-biserial 為正值時,代表流失客戶的交易金額較低。慣例: 0.1 弱、0.3 中、0.5 強。
- 兩組中位數:效果量的業務語言版本,「典型留存客戶的年消費是典型流失客戶的幾倍」。

In [6]:
# Cell M3-16(code)|T4 執行:total_trans_amt 流失組 vs 留存組,Mann-Whitney U 檢定(附 t 檢定對照)

churned_amt = df.loc[df["is_churned"], "total_trans_amt"]
retained_amt = df.loc[~df["is_churned"], "total_trans_amt"]
n1, n2 = len(churned_amt), len(retained_amt)

# 1. 描述統計:中位數與平均數並列,平均數與中位數的落差即右偏的證據(承接 notebook 02 Cell E-8)
desc = pd.DataFrame({
    "n": [n1, n2],
    "median": [churned_amt.median(), retained_amt.median()],
    "mean": [round(churned_amt.mean(), 1), round(retained_amt.mean(), 1)],
}, index=["churned", "retained"])
display(desc)

# 2. Mann-Whitney U 檢定(雙尾)。第一個參數為流失組,故 U 數的是「流失客戶交易金額較高」的配對數。
u_stat, p_t4 = stats.mannwhitneyu(churned_amt, retained_amt, alternative="two-sided")
total_pairs = n1 * n2
p_churn_higher = u_stat / total_pairs           # 流失客戶金額較高的配對比例
p_churn_lower = 1 - p_churn_higher              # 流失客戶金額較低的配對比例
rank_biserial = p_churn_lower - p_churn_higher  # rank-biserial 為正值時,代表流失客戶的交易金額較低,= 2 × p_churn_lower − 1

# 3. 對照:Welch t 檢定(不假設兩組變異數相等),用以說明 t 檢定在大樣本下結論一致,但判讀以 Mann-Whitney 為準([M3-3])
t_stat, p_t = stats.ttest_ind(churned_amt, retained_amt, equal_var=False)

print(f"總配對數 = {total_pairs:,};U = {u_stat:,.0f}(流失客戶交易金額較高的配對數),無差時應約 {total_pairs / 2:,.0f}")
print(f"流失客戶交易金額較低的配對比例 = {p_churn_lower:.4f};rank-biserial(正值代表流失客戶交易金額較低)= {rank_biserial:.3f}")
print(f"Mann-Whitney:p = {p_t4:.3g}, 校正後門檻 {ALPHA_ADJ} → 統計顯著: {p_t4 < ALPHA_ADJ}")
print(f"中位數比:留存 {retained_amt.median():,.0f} ÷ 流失 {churned_amt.median():,.0f} = {retained_amt.median() / churned_amt.median():.2f} 倍")
print(f"對照 Welch t 檢定:t = {t_stat:.2f}, p = {p_t:.3g}(結論一致,但平均數受右偏影響,判讀不採)")

,n,median,mean
churned,1627,2329.0,3095.0
retained,8500,4100.0,4654.7


總配對數 = 13,829,500;U = 4,481,880(流失客戶交易金額較高的配對數),無差時應約 6,914,750
流失客戶交易金額較低的配對比例 = 0.6759;rank-biserial(正值代表流失客戶交易金額較低)= 0.352
Mann-Whitney:p = 2.72e-112, 校正後門檻 0.0125 → 統計顯著: True
中位數比:留存 4,100 ÷ 流失 2,329 = 1.76 倍
對照 Welch t 檢定:t = -22.69, p = 6.35e-106(結論一致,但平均數受右偏影響,判讀不採)


Cell M3-17(markdown)|T4 判讀

**T4 判讀|近 12 個月交易金額**

- **實測**:流失組(n=1,627)中位數 2,329 元、平均數 3,095 元;留存組(n=8,500)中位數 4,100 元、平均數 4,655 元。Mann-Whitney U = 4,481,880(無差時應約 6,914,750),p ≈ 10⁻¹¹²;流失客戶交易金額較低的配對比例 0.676,rank-biserial = 0.352;中位數比 1.76 倍。對照的 Welch t 檢定 p ≈ 10⁻¹⁰⁶,結論一致。

- **判定**:統計顯著(p 遠低於校正後門檻 0.0125 與 α/9)。U 值 4,481,880 只有「無差時應有值」的 65%,代表流失客戶在絕大多數配對裡金額較低,這種偏離純靠抽樣波動幾乎不可能發生。效果量 rank-biserial = 0.352,屬中等:隨機抽一位流失客戶與一位留存客戶,流失客戶消費較低的機率約 68%;若兩組沒有差別,此機率應為 50%。實務顯著:典型留存客戶(中位數 4,100 元)的年消費是典型流失客戶(2,329 元)的 1.76 倍,差距足以讓交易金額在客群之間形成有意義的價值分層。

- **對決策的意涵**:<br/>
(1) 交易金額作為 M5 價值軸的前提成立:流失組與留存組在此變數上確實不同,且差距為中等以上,可用它把客戶分為高、中、低價值層。<br/>
(2) 兩組分布仍有重疊(流失客戶消費較低的配對比例為 68%,不是 100%),代表流失組中存在消費不低於一般留存客戶的客戶,約占三成;這群「高價值卻流失」的客戶正是 M5 搶救區(高風險 × 高價值)要優先辨識的對象,重疊的存在是價值軸有用的原因,不是缺陷。<br/>
(3) 判讀以 Mann-Whitney 為準而非 t 檢定:兩者 p 值都極小,但 t 檢定比較的平均數(3,095 vs 4,655)受少數高消費客戶影響,不代表典型客戶;中位數(2,329 vs 4,100)才是對決策者有意義的數字(決策日誌 [M3-3])。<br/>

- **待辦或限制**:<br/>
(1) 時序限制同 notebook 02 Cell E-8:快照資料無法區分「消費先下降、之後流失」與「決定流失後停止使用、因此看起來消費低」,因此交易金額是客戶價值的代理,不是流失的「預警訊號」;M5 以它排序價值,不以它預測風險。<br/>
(2) 交易金額是「近 12 個月」的彙總,流失客戶若在期間中途流失,其金額本身就會因使用期間縮短而偏低,這是代理變數的先天雜訊,M5 判讀價值層時須說明。<br/>
(3) M5 價值層的切點(高、中、低)本模組不決定,由 M5 依分位數與業務考量定案,新增待驗證事項#22(見第 11 節(Cell M3-23))。<br/>

Cell M3-18(markdown)|第 8 節:檢定力示範

## 第 8 節|檢定力示範:從效果量反推樣本數

**目的**:第 2 節 2.5 說明本模組四個檢定的檢定力都接近 1,不構成風險;本節用 H8 的實測效果量做兩件事:(1) 驗證此說法;(2) 示範「要檢出一個給定大小的差距,每組需要多少人」的計算,這是 M6 設計 A/B 測試時決定樣本數的方法,在此先以本案數字操作一次。

**計算的輸入**:效果量(此處用 Cohen's h,它把兩個比例的差距標準化成與人數無關的數字,T3 實測 h = 0.075)、顯著水準 α、目標檢定力(慣例 0.8)、兩組人數比例。給定其中三項,程式可解出第四項。

**三個問題**:
1. 以 H8 目前的樣本(女性 5,358 人、男性 4,769 人),檢定力是多少?
2. 若只想以 α = 0.05、檢定力 0.8 檢出 2.74 個百分點的差距,每組至少需要多少人?
3. 若樣本縮小為目前的十分之一,檢定力剩多少?(說明「同一個真實差距,樣本不夠時大多數情況測不出來」)

另附 M6 的實際情境試算:假設某挽留方案針對的客群基準流失率為 25%,希望檢出方案把流失率降到 20%(下降 5 個百分點)的效果,α = 0.05、檢定力 0.8,實驗組與對照組各需多少人。此數字供 M6 作為起點。

In [7]:
# Cell M3-19(code)|第 8 節執行:檢定力計算(statsmodels NormalIndPower,對應兩比例 z 檢定)

power_calc = NormalIndPower()
h_t3 = res_t3["cohen_h_raw"]             # T3 的 Cohen's h,計算值使用未四捨五入值(0.07488),避免四捨五入誤差傳入檢定力計算
n_f, n_m = res_t3["n_a"], res_t3["n_b"]  # 女性 5,358、男性 4,769
ratio = n_m / n_f                        # 兩組人數比

# 問題 1:目前樣本下的檢定力(分別以 α = 0.05 與校正後 0.0125 計算)
power_now_005 = power_calc.power(effect_size=h_t3, nobs1=n_f, ratio=ratio, alpha=ALPHA)
power_now_adj = power_calc.power(effect_size=h_t3, nobs1=n_f, ratio=ratio, alpha=ALPHA_ADJ)

# 問題 2:要以 α = 0.05、檢定力 0.8 檢出 h ≈ 0.0749 的差距,每組(兩組人數相同)需要多少人
n_needed_005 = power_calc.solve_power(effect_size=h_t3, power=0.8, alpha=ALPHA, ratio=1)
n_needed_adj = power_calc.solve_power(effect_size=h_t3, power=0.8, alpha=ALPHA_ADJ, ratio=1)

# 問題 3:樣本縮小為十分之一時的檢定力
power_tenth = power_calc.power(effect_size=h_t3, nobs1=n_f / 10, ratio=ratio, alpha=ALPHA)

# 對照:H4 的檢定力(h = 0.348,兩組 2,153 / 7,974)
power_t2 = power_calc.power(effect_size=res_t2["cohen_h_raw"], nobs1=res_t2["n_a"], ratio=res_t2["n_b"] / res_t2["n_a"], alpha=ALPHA_ADJ)

# M6 情境試算:基準 25% → 目標 20%,α = 0.05,檢定力 0.8,兩組人數相同
h_m6 = proportion_effectsize(0.25, 0.20)     # 基準 25% 在前、目標 20% 在後,使 h 為正值;樣本數計算只看絕對值,方向不影響結果
n_m6 = power_calc.solve_power(effect_size=h_m6, power=0.8, alpha=ALPHA, ratio=1)

print(f"H8(h = {h_t3:.4f}):目前樣本檢定力 α=0.05 → {power_now_005:.3f};α=0.0125 → {power_now_adj:.3f}")
print(f"H8:檢出此差距所需每組人數 α=0.05 → {n_needed_005:,.0f};α=0.0125 → {n_needed_adj:,.0f}")
print(f"H8:樣本縮為十分之一(女性 {n_f/10:.0f} 人)時檢定力 → {power_tenth:.3f}")
print(f"H4(h = {res_t2['cohen_h']}):目前樣本檢定力 α=0.0125 → {power_t2:.4f}")
print(f"M6 情境(25% → 20%,h = {h_m6:.3f}):每組所需人數 → {n_m6:,.0f}")

H8(h = 0.0749):目前樣本檢定力 α=0.05 → 0.964;α=0.0125 → 0.897
H8:檢出此差距所需每組人數 α=0.05 → 2,799;α=0.0125 → 3,977
H8:樣本縮為十分之一(女性 536 人)時檢定力 → 0.221
H4(h = 0.348):目前樣本檢定力 α=0.0125 → 1.0000
M6 情境(25% → 20%,h = 0.120):每組所需人數 → 1,092


Cell M3-20(markdown)|第 8 節判讀

**第 8 節判讀|檢定力**

- **實測**:H8 目前樣本的檢定力在 α = 0.05 下為 0.964、校正後 α = 0.0125 下為 0.897;要以 α = 0.05、檢定力 0.8 檢出 2.74 個百分點的差距,每組需 2,799 人(校正後門檻需 3,977 人);樣本縮小為十分之一時檢定力降至 0.221。H4 的檢定力為 1.0000。M6 情境(25% → 20%)每組需 1,092 人。

- **判定**:第 2 節 2.5 的說法獲得驗證:本模組差距最小的檢定(H8)檢定力仍有 0.90,其餘三個接近 1,「真的有差卻測不出來」的情況在本模組不會發生。H8 縮小十倍的對照說明了檢定力的意義:同樣 2.74 個百分點的真實差距,若女性只有 536 人、男性 477 人,十次檢定只有約兩次會得到顯著結果,其餘八次會「不顯著」;此時「不顯著」只代表樣本不足以看見差距,不代表差距不存在。

- **對決策的意涵**:<br/>
(1) M6 設計 A/B 測試時,樣本數必須在實驗前依「預期效果、α、目標檢定力」計算,不能事後補;以本案數字,若挽留方案的目標是把某客群流失率從 25% 降到 20%,實驗組與對照組各需約 1,100 人,總計約 2,200 人。若目標客群(例如淺關係且低活躍的客戶)人數不足,M6 須調整目標效果或延長實驗期間。<br/>
「若目標客群人數不足」:假設 M5 最後決定方案只針對「淺關係且低活躍」的客戶,這群人在資料裡只有 547 人,湊不到 2,200 人。此時有三種可行的調整:(a) 提高希望檢出的效果:例如目標改為 25% 降到 15%,需要的人數會少很多;(b) 延長實驗期間累積人數:讓陸續新進入該客群的客戶累積到足夠人數;(c) 放寬目標客群定義。<br/>
這三個選項各有代價,是 M6 要決定的事,第 8 節只負責提醒要「先算人數,再決定做不做得到」。<br/>
(2) 校正後門檻會提高所需樣本數(H8 由 2,799 升至 3,977),M6 若同時檢定多個指標,樣本數計算須以校正後的 α 為準。

- **待辦或限制**:此處的樣本數計算假設兩組人數相同且各觀察值獨立,M6 若採用「兩組人數不同的分組」或「分層隨機化」,計算方式須對應調整;新增待驗證事項#23(見第 11 節 (Cell M3-23))。


- **M6樣本數計算 補充說明**:
1. M6 要設計的實驗是:從某個目標客群(例如淺關係且低活躍的客戶)隨機分成兩組,一組收到挽留方案(實驗組),一組不收到(對照組),過一段時間後比較兩組的流失率。如果實驗組流失率明顯低於對照組,就稱「該方案有效」。
2. 注意:實驗組和對照組各要放多少人:人太少,即使方案真的有效,兩組流失率的差距也會被抽樣波動掩蓋,實驗做完得到 p = 0.3,但分不出是「方案沒效」還是「樣本數不夠所以看不出來」。因此,需要的樣本人數必須在實驗前算好(意即前述的「不能事後補」);事後補人再測一次還有另一個問題:每多檢定一次就多一次假陽性的機會(多重比較問題)。
3. 算人數需要:
- 四個輸入,本節用的假設值是:
(1)對照組的預期流失率:25%(假設的情境數字,代表一個高風險客群;實際 M6 會用目標客群的實測流失率)。
(2)希望能檢出的最小效果:方案把流失率從 25% 降到 20%,也就是降 5 個百分點 (例如:業務只在乎「降超過 5 個百分點的方案」)。
(3)α = 0.05。
(4)檢定力 0.8:慣例值。
- 輸出:每組 1,092 人,兩組合計約 2,200 人。
意思:如果方案真的能把流失率從 25% 降到 20%,兩組各放 1,092 人,實驗結束時有 80% 的機會得到顯著結果;放更少人,這個機會就往下掉。以第 2 節的標準誤說明:兩組各 1,092 人時,差距的標準誤約 1.8 個百分點,5 個百分點的差距約為標準誤的 2.8 倍,p 值落在顯著範圍的機率約八成。

Cell M3-21(markdown)|第 9 節:M3 結果總表

## 第 9 節|四個檢定的結果總表

| 檢定 | 對象與方法 | 統計量 | p 值 | 校正後(α = 0.0125)統計顯著 | 效果量 | 實務顯著 | 對決策的意涵 |
|---|---|---|---|---|---|---|---|
| T1 | H6 聯繫次數(0–6 次)× 流失;卡方獨立性檢定 | χ² = 586.6,自由度 6 | ≈ 10⁻¹²³ | 是 | Cramér's V = 0.241(弱至中) | 取決於預警線位置(判準:各組對基準的 diff_pp ≥ +9;4 次組 +6.56 未達,5 次組 +17.45、6 次組 +83.93 達到;Cramér's V 0.241 只描述整體關聯強度,不用於此判定) | 聯繫次數可作為風險標記(非提前預警);預警線由 M5 定案(#10-a) |
| T2 | H4 往來產品數(≤2 vs ≥3)× 流失;兩比例 z 檢定 | z = 15.42 | ≈ 10⁻⁵³ | 是 | 比例差 13.75 pp(95% CI 11.74 至 15.76);勝算比 2.43 | 是(比例差 13.75 pp ≥ 9;CI 下限 11.74 仍 ≥ 9;Cohen's h 0.348 高於弱效果門檻 0.2) | 淺關係旗標作為 M5 名單條件的前提成立;名單條件應為「淺關係且低活躍」;深化策略的關聯成立,因果待 M6 實驗驗證 |
| T3 | H8 性別(女 vs 男)× 流失;兩比例 z 檢定 | z = 3.75 | 0.00018 | 是 | 比例差 2.74 pp(95% CI 1.32 至 4.17)(上限 4.17 也遠低於 +9 的工作判準);勝算比 1.23;Cohen's h = 0.075(低於弱效果的慣例門檻 0.2) | 否(比例差 2.74 pp 遠低於 9;CI 上限 4.17 仍低於 9;Cohen's h 0.075 低於弱效果門檻 0.2) | 統計顯著但實務不顯著的示範;不設計性別導向方案;M7 公平性檢視的基準 |
| T4 | total_trans_amt 流失 vs 留存;Mann-Whitney U 檢定 | U = 4,481,880 | ≈ 10⁻¹¹² | 是 | rank-biserial = 0.352(中);中位數 2,329 vs 4,100(1.76 倍) | 是(中位數比 1.76 倍,典型客戶差距足以形成價值分層;rank-biserial 0.352 達中等門檻 0.3;本檢定比較金額而非流失率,判準用效果量與中位數,不用百分點差距) | 交易金額作為 M5 價值軸的前提成立;分布重疊的三成即搶救區(高風險 x 高價值)的來源 |

**總表的三個結論**:
1. 四個檢定全部統計顯著,且 p 值皆遠低於更嚴格的 α/9 ≈ 0.0056,檢定對象的挑選方式不影響任何結論(第 1 節 1.4)。<br/>
2. 統計顯著與實務顯著分離的案例有兩個:H8 差距太小,不值得行動;H6 的實務顯著取決於預警線劃在幾次,是商業決策而非統計問題。這兩個案例說明 M5 的名單與策略不能只依據 p 值。<br/>
3. 四個檢定確認的是「關聯真實存在且大小如何」,沒有一個能證明因果:H4 的深化策略、H6 的預警介入是否有效,只有 M6 的實驗能回答。

**實務顯著判準(決策日誌 [M3-4])**:<br/>
1. 看該檢定所比較的那兩個數字之差是否 ≥ +9 個百分點:(詳細可見 Cell M3-3 2.2 效果量與信賴區間)<br/>
舉例:兩比例z檢定比較的是兩組別的差距(女 vs 男、淺 vs 非淺),所以判準看「組間差距」。卡方檢定的判準看「每一組對基準的 diff_pp」。<br/>
原則:p 值不參與實務顯著的判定。<br/>
各組判準:<br/>
(1) 兩組比較(T2、T3)看組間差距,並以信賴區間的下限(或上限)確認最保守情況是否仍成立,再以 Cohen's h 對照慣例門檻 0.2;<br/>
(2) 多組比較(T1)看各組對基準的 diff_pp;<br/>
(3) 連續型檢定(T4)看效果量是否達中等(0.3)並以中位數說明業務意義。<br/>
2. 此判準為本案自訂,最終是否行動由 M5 結合成本效益定案。

**補充**:<br/>
T3 業務語言:若依性別篩選做挽留名單,圈到的女性客戶流失率 17.36%,只比全體 16.07% 高 1.29 個百分點。這份挽留名單與「隨機挑人」幾乎沒有差別,投放挽留資源沒有意義。<br/>
解釋:<br/>
1. 挽留名單:從一萬名客戶裡挑一部分人出來投放挽留資源。名單是否有效益,看的是「名單裡真的會流失的人占幾成」,也就是名單的流失率。若名單的流失率是 20%,表示投下去的資源有 20% 花在真的會流失的人身上。
2. 若用性別做挽留名單:「圈出全部女性」,因女性流失率為 17.36%, 所以把所有女性圈出來的這份名單,流失率就是 17.36%。
若用全體客戶做挽留名單(不篩選性別、完全隨機):這份挽留名單的流失率是 16.07% (全體客戶的流失率),因為隨機抽樣本測出的流失率會和全體流失率相同。
3. 兩者相減 = 1.29 個百分點,表示: 若用性別篩選,比起不篩選,名單流失率只提高 1.29 個百分點。名單流失率 17.36% 的狀況下,這份挽留名單裡 82.6% 的人本來就不會流失,資源一樣大量投在不會流失的人身上(82.6%),與隨機挑人幾乎一樣(不會流失的人 84%)。
4. 因此,用性別做篩選圈出的挽留名單,並針對該份名單投放挽留資源「沒有意義」。

Cell M3-22(markdown)|第 10 節:方法限制

## 第 10 節|方法限制

1. **觀察性資料只能建立關聯**:本模組四個檢定使用的是客戶快照資料,沒有任何條件是分析者介入操控的,因此檢定通過只代表「兩變數的關聯不是抽樣波動」,不代表「若主動改變其中一個變數(例如幫客戶增加往來產品數),另一個變數(流失率)就會跟著改變」。淺關係客戶流失率高,不等於幫客戶增加產品就能降低流失;聯繫次數多者流失率高,不等於減少聯繫就能減少流失。因果驗證交由 M6 的實驗設計,M8 報告的措辭須嚴格區分「有關聯」與「是造成流失的原因」。
2. **快照資料的時序問題**:所有欄位都是快照時點的回溯彙總值,沒有時間戳記(問題定義書 §5),無法判斷變數的變化發生在流失決定之前或之後。此限制對 H6(聯繫可能是流失手續的痕跡)與 total_trans_amt(金額可能因流失後停用而偏低)影響最大,兩者的定位因此分別是「風險標記」與「價值代理」,而非「預警訊號」。
3. **卡方檢定不檢驗順序**:T1 證明聯繫次數與流失有關聯,但「聯繫越多次、流失率越高」的趨勢由 M2 切片曲線描述,不是「卡方檢定的結論」;若 M8 要陳述趨勢,引用的是切片數字而非 χ²。
4. **Bonferroni 校正偏保守**:校正後門檻使假陰性風險上升,本案因四個差距都很大而不受影響;但若後續模組(如 M6 同時檢定多個指標)遇到差距較小的情境,應重新評估是否改用 Holm 法或 FDR 法。
5. **實務顯著的判準為本案自訂**:+9 個百分點的工作判準取自候選洞察清單的分層門檻,不是通用標準;缺少挽留成本與客戶利潤資料時,它是排序用的替代品,最終行動與否須由 M5 以成本效益定案。
6. **檢定的分組沿用 M2 的界線**:H4 的 ≤2 項、H6 的逐值分組都是 M2 依實測斷點決定的;檢定確認的是「以這些界線分組時,關聯成立」,不是「這些界線是最佳切點」。切點的最佳化(如 H3 的門檻位置,待驗證事項 #17)不在本模組範圍。

Cell M3-23(markdown)|第 11 節:待驗證事項清單 v4 與 M4 開工檢核

## 第 11 節|待驗證事項清單 v4 (M3 收尾定版)

**定位**:承接 notebook 02 Cell E-9 的 v3,回填本模組承接的 #12、#15、#16,並新增本模組產生的待辦(#19 至 #23)。編號沿用,不重編。後續每個模組開工時依「交辦階段」欄篩出承接項目,收尾時回填狀態。

**已結案項目**(於 M2 內完成驗證,列出以保留證據,後續不需再處理):

| # | 事項 | 驗證結果 | 依據 |
|---|---|---|---|
| 1 | H1 的 Q2>Q1 預期外形狀:「衰退方向比活躍水位更能預警」假說 | ✅ 假說被推翻:Q1(10–41 筆)與 Q2(41–61 筆)的消費變化比分布高度重疊,中位數 0.694 vs 0.738;兩箱為同一性質客群,M3/M5 以低活躍旗標(≤61 筆)合併處理;Q2 流失率高於 Q1 的成因,本資料無其他欄位可查,列為原因未明 | notebook 02 Cell E-3/E-4(E1 判讀) |
| 2 | 零動用訊號的精確 diff_pp 幅度(0 vs >0 二分重驗) | ✅ n=2,470、流失率 36.15%、diff_pp +20.08;清單 v2 起一律引用此數字 | notebook 02 Cell C-3/C-4(C1 判讀) |
| 3 | 活躍水位類訊號的名單重疊規模 | ✅ 四個正式旗標名單合計 10,410 人次,去重後 7,292 人(重複 30%);聯集占全體 72%,「至少命中一旗標」不具篩選功能 | notebook 02 Cell C-7/C-8(C3-a 判讀) |
| 4 | 低額度訊號的獨立性(是否為收入效應的影子) | ✅ 控制收入後六組組內差距全部同向(+3.1 ~ +12.9 pp),低額度為獨立訊號;升級進候選洞察清單 v2(觀察層),列入 M4 候選特徵 | notebook 02 Cell C-11/C-12(C4 判讀) |
| 5 | 消費下滑(方向類訊號)與低活躍(水位類訊號)的重疊程度與性質 | ✅ 人數重疊 60%(1,211 / 2,012);性質問題由 E1 結案:Q1/Q2 為同一性質客群 | notebook 02 Cell C-8 第四點、Cell E-4 |
| 10 | avg_utilization_ratio 定義驗證(= total_revolving_bal / credit_limit?) | ✅ mismatch = 0,定義實證成立,證據等級升為「已驗證」;total_amt_chng_q4_q1 無法驗證,維持「推斷」 | notebook 02 Cell 35.1/35.2、[M2-4] |


**本模組結案的項目**:

| # | 事項 | 結案結果 | 依據 |
|---|---|---|---|
| 12 | 事後發現二則(Doctorate、Unknown × 低額度)是否進 M3 檢定 | ✅ 不進檢定:以產生假設的同一份資料檢定屬循環論證,維持「事後發現,列為觀察」;若 M4 模型顯示教育程度或收入 Unknown 的係數顯著,再以模型結果對照 | notebook 03 第 1 節 1.1、決策日誌 [M3-0] |
| 16 | H8 性別差距作為「統計顯著 vs 實務顯著」的示範案例 | ✅ 完成:p = 0.00018 統計顯著,差距 2.74 個百分點、Cohen's h = 0.075 實務不顯著;不設計性別導向方案 | notebook 03 T3 判讀 |
| 15(M3 部分) | H6 因果方向未定,M3 檢定報告須明示「風險標記,非提前預警」 | ✅ M3 部分完成:T1 判讀已明示;M8 簡報部分保留(見下表) | notebook 03 T1 判讀 |

**交辦後續模組的項目**(v3 既有項目維持原文,本模組新增 #19 至 #23):

| # | 事項 | 為什麼要做 | 交辦階段 | 依據 |
|---|---|---|---|---|
| 6 | 建模時評估納入交互項(low_activity 分別乘上零動用 / 淺關係 / 高聯繫) | C2 效果分解:同一訊號在低活躍與活躍正常子群內的效果相差 2.4–15 倍,模型需能表達此結構 | M4 | notebook 02 Cell C-6、[M2-8] |
| 7 | 檢查旗標特徵之間的相關性對迴歸係數穩定性的影響 | 共線性:兩個高度相關的變數放入同一迴歸模型,模型難以分辨影響歸屬,係數估計不穩定;由 C3-a 的高度重疊可預期旗標轉為特徵後彼此高度相關 | M4 | notebook 02 Cell C-8 |
| 8 | M4 模型評估報告必須附上與 C3-b 命中數分層基準的成績比較 | 命中 ≥3 名單精準度已達 83%,未用任何模型;模型須明顯勝過此基準才有引入的理由 | M4 | notebook 02 Cell C-10、[M2-9] |
| 18 | tenure 進模型的兩項注意:(1) 與 customer_age 相關係數 0.7889,須二擇一;(2) tenure=36 單值聚集 2,463 人(占 25–36 箱 45%),使其有效區分力低於值域表面所示;年齡 × tenure 的交互作用單變數切片無法驗證,若任一進模型由模型評估 | 避免共線導致係數不穩,並避免把 tenure 的低重要性誤讀為「年資不重要」 | M4 | notebook 02 Cell 31、Cell D-4、sql/03_cohort.sql [D1-補]、[M1-5] |
| 19 | 6 次組流失率 100%(n=54)在 logistic regression 中會造成「完全分離」:該組所有客戶結果相同,模型無法穩定估計此類別的係數 | 建模前須決定處理方式:將 5 次與 6 次合併為「≥5 次」,或以連續變數形式納入聯繫次數而非逐次類別 | M4 | notebook 03 T1 判讀 |
| 20 | M4 的 logistic regression 係數以勝算比解讀時,與 T2 單變數勝算比(2.43)對照:多變數模型中淺關係的勝算比若明顯縮小,代表其效果部分被低活躍等其他變數解釋 | 檢查單變數關聯在多變數控制後是否維持,是模型解讀的基本步驟 | M4 | notebook 03 T2 判讀 |
| 9 | 挽留名單門檻(命中旗標數 ≥2 或 ≥3)依預算與成本效益定案 | 覆蓋率與精準度無法兩全,屬商業決策 | M5 | notebook 02 Cell C-10、[M2-9] |
| 10-a | 高聯繫預警線位置(現為 ≥4 次的工作預設值)依成本效益定案 | T1 證明關聯真實但不指出門檻;4 次組 diff_pp +6.56 未達實務判準,5 次以上遠超過但人數僅 230 人 | M5 | notebook 02 Cell 21 註、notebook 03 T1 判讀、[M2-5] |
| 17 | H3 消費變化比的門檻位置:0.604 僅為五分位箱界,非精確門檻 | 若 M5 採「變化比 < X 即示警」規則,X 需要更準的定位:對 0.4–0.8 區間做更細分箱(如十分位),觀察流失率跳升的確切位置 | M5 | notebook 02 Cell 27 |
| 21 | M5 名單條件應為「淺關係且低活躍」,不以「淺關係」單獨成立;估算預期效益時以 T2 信賴區間下限 11.74 個百分點做保守估計 | T2 通過的是全體平均差距,C2 顯示效果集中於低活躍子群;信賴區間下限為保守估算的依據 | M5 | notebook 03 T2 判讀、notebook 02 Cell C-6 |
| 22 | M5 價值軸以 total_trans_amt 分高、中、低價值層,切點(如三分位或依業務金額)由 M5 定案;判讀時說明「近 12 個月彙總值對中途流失客戶偏低」的代理雜訊 | T4 確認交易金額能區分客群,但價值層切點與代理雜訊的說明不在 M3 範圍 | M5 | notebook 03 T4 判讀 |
| 11 | 「調高額度」不得直接列為挽留手段;若要採用,須經實驗驗證 | 低額度與流失的機制未定:(a) 額度造成使用不便,或 (b) 額度只是關係深度的痕跡;若為 (b),調額是處理痕跡而非成因,唯有 M6 實驗能驗證 | M5 / M6 | notebook 02 Cell C-12 |
| 15(M8 部分) | M8 簡報陳述 H6 時須明示「風險標記,非提前預警」,不得寫成因果 | 快照資料無法區分聯繫與流失決定的先後 | M8 | notebook 03 T1 判讀、[M2-5] |
| 23 | M6 樣本數計算沿用第 8 節方法(Cohen's h、α、檢定力 0.8);若同時檢定多個指標,以校正後 α 計算;「兩組人數不同的分組」或「分層隨機化」時調整計算方式 | 實驗前必須決定樣本數,事後無法補救 | M6 | notebook 03 第 8 節 |
| 13 | mart 層匯出:依 M8 儀表板設計定案後,一次匯出 data/processed/ 的 CSV | 下游需求(儀表板要呈現什麼)未定前匯出無驗收標準;只是延後,不是取消 | M8 | [M2-13] |
| 14 | Excel 樞紐分析重現:以樞紐表重現一張儀表板選用的切片,驗證與 SQL 數字一致,產出 churn_pivot_summary.xlsx,README 技術棧列入 Excel | 展示跨工具重現同一分析並核對數字一致的能力;與 #13 一併完成 | M8 | [M2-14] |

**清單演變紀錄**:
v3(M2 收尾,#1–#18) → v4 (本表) <br/>
本表:<br/>
- 結案:#12、#16
- #15 拆為 M3 部分(結案)與 M8 部分(保留)
- 新增: #19 – #23

**M4 開工檢核**(進入 M4 前先核對):
- 承接 #6、#7、#8、#18、#19、#20
- M4 收尾時回填六者狀態,並將本表複製至 notebook 04 開頭作為承接依據。